# EkaQuant Master Evaluation Baseline\n**Guarantees:**\n1. **GPU Enforcement:** Aborts if not T4.\n2. **Clean Alignment:** No SyntaxErrors.\n3. **Isolated Results:** Timestamped folders.

In [ ]:
import os
import subprocess
import sys
from datetime import datetime
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# 1. Pre-flight Hardware Check
print('--- Hardware Verification ---\n')
try:
    gpu_info = subprocess.check_output('nvidia-smi -L', shell=True).decode()
    print(gpu_info)
    if 'T4' not in gpu_info:
        print('FATAL ERROR: This notebook requires T4 GPUs for NF4 support. Detected P100 or other.')
        print('Stopping execution to save quota.')
        sys.exit(1)
except Exception as e:
    print(f'GPU check failed: {e}')
    sys.exit(1)

RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
print(f'Run ID: {RUN_ID}')

# 2. Hugging Face Authentication
user_secrets = UserSecretsClient()
try:
    hf_token = user_secrets.get_secret('HF_TOKEN')
    if hf_token:
        login(token=hf_token)
except Exception as e:
    print(f'Auth Warning: {e}')

In [ ]:
print('--- Clean Installation ---\n')
subprocess.run('rm -rf eka-eval EkaQuant', shell=True)
subprocess.run('pip install -q transformers bitsandbytes accelerate peft datasets numpy scipy kneed scikit-image tqdm', shell=True)
subprocess.run('git clone https://github.com/lingo-iitgn/eka-eval.git', shell=True)
os.chdir('eka-eval')
subprocess.run('pip install -q evaluate rouge_score', shell=True)
subprocess.run('pip install -e .', shell=True)
os.chdir('..')
subprocess.run('rm -rf eka-eval/results_output eka-eval/results', shell=True)

In [ ]:
def run_eval(model_id, precision_bit):
    model_name = model_id.split('/')[-1]
    target_dir = f'/kaggle/working/eval_{RUN_ID}_{precision_bit}bit_{model_name}'
    os.makedirs(target_dir, exist_ok=True)
    print(f'\n[STARTING] {precision_bit}-bit eval for {model_id}')
    
    loader_path = 'eka-eval/eka_eval/core/model_loader.py'
    config_path = 'eka-eval/eka_eval/config/benchmark_config.py'
    
    # Patch precision
    if precision_bit == 8:
        subprocess.run(['sed', '-i', 's/load_in_4bit *= *True/load_in_8bit=True/g', loader_path])
    else:
        subprocess.run(['sed', '-i', 's/load_in_8bit *= *True/load_in_4bit=True/g', loader_path])
    
    # Patch Typo
    subprocess.run(['sed', '-i', 's/indic\.mmlu_in\.evaluate_mmlu_in/multilingual.mmlu_in.evaluate_mmlu_in/g', config_path])
    
    # Execute with precise prompt sequence
    # 1 (Local) -> 1 (HF) -> {model_id} -> no (BM) -> 9 (INDIC) -> 1 (MMLU-IN) -> no (viz)
    wizard_input = f'1\n1\n{model_id}\nno\n9\n1\nno\n'
    subprocess.run('python eka-eval/scripts/run_benchmarks.py', input=wizard_input, shell=True, text=True)
    
    if os.path.exists('eka-eval/results_output'):
        subprocess.run(f'cp -r eka-eval/results_output/* {target_dir}/', shell=True)
        print(f'[SUCCESS] Results moved to {target_dir}')
    else:
        print(f'[ERROR] No results found in eka-eval/results_output!')

model_id = 'Qwen/Qwen2.5-7B-Instruct'
run_eval(model_id, 8)
run_eval(model_id, 4)

In [ ]:
import glob
import pandas as pd
print('
--- Final Result Summary ---\n')
csv_files = sorted(glob.glob(f'/kaggle/working/eval_{RUN_ID}_*bit_*/calculated.csv'))
for csv in csv_files:
    print(f'\nFile: {csv}')
    try:
        df = pd.read_csv(csv)
        print(df.to_markdown())
    except Exception as e:
        print(f'Error reading CSV: {e}')

zip_name = f'/kaggle/working/results_{RUN_ID}.zip'
print(f'\nZipping results into {zip_name}...')
subprocess.run(f'zip -r {zip_name} /kaggle/working/eval_{RUN_ID}_*', shell=True)